# TinyLlama-1.1B LoRA — Optuna Tuning (Notebook 2 of 2)

Reads `shared_config.json` + `baseline_final_metrics.csv` from the **baseline notebook's output**
(added as an input dataset) to reuse the exact same train/val split and fallback values.

Runs a reduced, resumable Optuna study (tuning-subsample + 5 trials + 2 fixed epochs/trial),
then retrains the best config on full train+val and evaluates on test once.

**Before running:** in the Kaggle sidebar, click **Add Input → Notebooks**, select your saved
baseline notebook, and confirm `BASELINE_INPUT_DIR` below matches the mounted path.


In [ ]:
%pip install -q transformers datasets accelerate peft scikit-learn pandas matplotlib optuna
%pip uninstall -y torchao -q


In [ ]:
import os, re, time, json, math, random, gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import optuna
from datasets import Dataset

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback
)
from peft import LoraConfig, TaskType, get_peft_model


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ------------------------------------------------------------------
# UPDATE THIS to match the mounted path shown after Add Input -> Notebooks
# ------------------------------------------------------------------
BASELINE_INPUT_DIR = "/kaggle/input/tinyllama-1-baseline"   # <- rename to your actual notebook slug

with open(f"{BASELINE_INPUT_DIR}/shared_config.json") as f:
    shared = json.load(f)

SERIALIZATION = shared["serialization"]
fallback_log_target = shared["fallback_log_target"]
max_reasonable_log_target = shared["max_reasonable_log_target"]
train_ids = set(shared["train_ids"])
val_ids = set(shared["val_ids"])

# NOTE: tinyllama_baseline.ipynb does not write train_file/test_file into
# shared_config.json -- it hardcodes these paths in its own config cell, so we
# mirror that here directly instead of reading them from `shared`.
DATA_DIR = "/kaggle/input/datasets/tamislam/llm-serialisation-dataset-final/kickstarter_serializations/"
TRAIN_FILE = f"{DATA_DIR}/kickstarter_llm_train.csv"
TEST_FILE  = f"{DATA_DIR}/kickstarter_llm_test.csv"

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "/kaggle/working"
OPTUNA_WORKDIR = f"{OUTPUT_DIR}/optuna_trials"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/tuned_best_model"
STUDY_DB_PATH = f"{OUTPUT_DIR}/optuna_study.db"

MAX_LENGTH = 512
MAX_NEW_TOKENS = 12
GENERATION_BATCH_SIZE = 32

SYSTEM_PROMPT = (
    "Predict the final Kickstarter funding value as log1p(USD) "
    "from the provided campaign features. Output only the numeric value."
)

# --- Speed levers (kept small to fit a 12hr Kaggle session with margin) ---
N_TRIALS = 5
FIXED_EPOCHS_PER_TRIAL = 2
TUNING_SUBSAMPLE_SIZE = 5500   # rows sampled from train_ids for trial training only

print("Serialization (from baseline handoff):", SERIALIZATION)
print("N_TRIALS:", N_TRIALS, "| epochs/trial:", FIXED_EPOCHS_PER_TRIAL, "| tuning subsample:", TUNING_SUBSAMPLE_SIZE)


## Rebuild the EXACT same train/val split as the baseline notebook (from shared IDs)

In [ ]:
train_full_df = pd.read_csv(TRAIN_FILE, keep_default_na=False)
test_df = pd.read_csv(TEST_FILE, keep_default_na=False)

for col in ["target", "target_usd"]:
    train_full_df[col] = pd.to_numeric(train_full_df[col], errors="raise")
    test_df[col] = pd.to_numeric(test_df[col], errors="raise")

train_df = train_full_df[train_full_df["id"].isin(train_ids)].reset_index(drop=True)
val_df = train_full_df[train_full_df["id"].isin(val_ids)].reset_index(drop=True)

assert len(train_df) == len(train_ids), "Train split mismatch vs shared_config -- check TRAIN_FILE matches baseline's."
assert len(val_df) == len(val_ids), "Val split mismatch vs shared_config -- check TRAIN_FILE matches baseline's."

# Subsample ONLY for Optuna trials -- final retrain below uses full train_df + val_df
rng = np.random.default_rng(SEED)
subsample_n = min(TUNING_SUBSAMPLE_SIZE, len(train_df))
tuning_train_df = train_df.sample(n=subsample_n, random_state=SEED).reset_index(drop=True)

print("Full train rows:", len(train_df), "| Val rows:", len(val_df), "| Test rows:", len(test_df))
print("Tuning subsample rows (trials only):", len(tuning_train_df))


## Tokenizer & shared helper functions (identical logic to baseline notebook)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_bf16_supported():
    TRAIN_DTYPE, USE_BF16, USE_FP16 = torch.bfloat16, True, False
else:
    TRAIN_DTYPE, USE_BF16, USE_FP16 = torch.float16, False, True
print("dtype:", TRAIN_DTYPE)


def build_training_example(example):
    target_text = f"{float(example['target']):.4f}"
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(example["text"])},
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": target_text}]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    full = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    input_ids = full["input_ids"]; attention_mask = full["attention_mask"]
    labels = input_ids.copy()
    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


class CausalLMDataCollator:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        batch_input_ids, batch_attention_mask, batch_labels = [], [], []
        for item in features:
            pad_len = max_len - len(item["input_ids"])
            batch_input_ids.append(item["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            batch_attention_mask.append(item["attention_mask"] + [0] * pad_len)
            batch_labels.append(item["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }

data_collator = CausalLMDataCollator(tokenizer)
NUMBER_PATTERN = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")

def parse_generated_log_target(text):
    match = NUMBER_PATTERN.search(str(text))
    if match is None: return np.nan
    try: value = float(match.group(0))
    except Exception: return np.nan
    if not np.isfinite(value) or value < 0 or value > max_reasonable_log_target: return np.nan
    return value

def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd):
    return {
        "MAE_log": mean_absolute_error(y_true_log, pred_log),
        "MSE_log": mean_squared_error(y_true_log, pred_log),
        "RMSE_log": rmse(y_true_log, pred_log),
        "R2_log": r2_score(y_true_log, pred_log),
        "MAE_USD": mean_absolute_error(actual_usd, pred_usd),
        "MSE_USD": mean_squared_error(actual_usd, pred_usd),
        "RMSE_USD": rmse(actual_usd, pred_usd),
        "R2_USD": r2_score(actual_usd, pred_usd),
        "RMSLE": np.sqrt(mean_squared_log_error(actual_usd, pred_usd)),
    }

def build_inference_prompt(feature_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": str(feature_text)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def generate_predictions(model, eval_df, batch_size=GENERATION_BATCH_SIZE):
    model.eval(); model.config.use_cache = True; tokenizer.padding_side = "left"
    device = next(model.parameters()).device
    prompts = [build_inference_prompt(x) for x in eval_df["text"]]
    generated_texts = []
    t0 = time.perf_counter()
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        prompt_width = inputs["input_ids"].shape[1]
        new_tokens = generated_ids[:, prompt_width:]
        generated_texts.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    gen_time = time.perf_counter() - t0
    pred_log_raw = np.array([parse_generated_log_target(t) for t in generated_texts], dtype=float)
    valid_mask = np.isfinite(pred_log_raw)
    pred_log = pred_log_raw.copy(); pred_log[~valid_mask] = fallback_log_target
    tokenizer.padding_side = "right"; model.config.use_cache = False
    return pred_log, valid_mask, generated_texts, gen_time


## Per-epoch metric logger + train/eval routine with checkpointing (same pattern as baseline)

In [ ]:
class EpochMetricsCallback(TrainerCallback):
    def __init__(self, eval_df, metrics_csv_path):
        self.eval_df = eval_df
        self.metrics_csv_path = metrics_csv_path
        self.rows = []

    def on_epoch_end(self, args, state, control, **kwargs):
        model = kwargs["model"]
        pred_log, valid_mask, _, gen_time = generate_predictions(model, self.eval_df)
        y_true_log = self.eval_df["target"].to_numpy(dtype=float)
        actual_usd = self.eval_df["target_usd"].to_numpy(dtype=float)
        pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)
        metrics = evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd)
        metrics["epoch"] = round(state.epoch, 4)
        metrics["parse_success_rate"] = float(valid_mask.mean())
        metrics["generation_time_seconds"] = gen_time
        self.rows.append(metrics)
        pd.DataFrame(self.rows).to_csv(self.metrics_csv_path, index=False)
        print(f"  [epoch {metrics['epoch']}] RMSE_log={metrics['RMSE_log']:.4f} "
              f"R2_log={metrics['R2_log']:.4f} parse_rate={metrics['parse_success_rate']*100:.1f}%")
        model.train()
        return control


def make_model_with_lora(params):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=TRAIN_DTYPE)
    model.config.use_cache = False
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, inference_mode=False,
        r=params["r"], lora_alpha=params["lora_alpha"], lora_dropout=params["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    return get_peft_model(model, lora_config)


def tokenize_split(df):
    hf_ds = Dataset.from_pandas(df[["text", "target"]], preserve_index=False)
    return hf_ds.map(build_training_example, remove_columns=hf_ds.column_names)


def train_and_eval(params, train_df_local, eval_df_local, output_dir, run_label="", epoch_metrics_csv=None):
    cleanup_gpu()
    model = make_model_with_lora(params)
    tokenized_train = tokenize_split(train_df_local)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=params["num_train_epochs"],
        per_device_train_batch_size=params["train_batch_size"],
        gradient_accumulation_steps=params["gradient_accumulation_steps"],
        learning_rate=params["learning_rate"],
        warmup_ratio=params["warmup_ratio"],
        weight_decay=params["weight_decay"],
        lr_scheduler_type=params["lr_scheduler_type"],
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=1,   # trials are disposable -- keep only latest checkpoint
        eval_strategy="no",
        bf16=USE_BF16, fp16=USE_FP16,
        optim="adamw_torch",
        report_to="none",
        remove_unused_columns=False,
        seed=SEED, data_seed=SEED,
        disable_tqdm=True,
    )

    callbacks = []
    if epoch_metrics_csv is not None:
        callbacks.append(EpochMetricsCallback(eval_df_local, epoch_metrics_csv))

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=tokenized_train, data_collator=data_collator,
        callbacks=callbacks,
    )

    resume_ckpt = None
    if os.path.isdir(output_dir):
        existing = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
        if existing:
            resume_ckpt = True
            print(f"Found existing checkpoint(s) in {output_dir} -> resuming training.")

    torch.cuda.empty_cache()
    t0 = time.perf_counter()
    trainer.train(resume_from_checkpoint=resume_ckpt)
    training_time = time.perf_counter() - t0

    pred_log, valid_mask, generated_texts, gen_time = generate_predictions(trainer.model, eval_df_local)
    y_true_log = eval_df_local["target"].to_numpy(dtype=float)
    actual_usd = eval_df_local["target_usd"].to_numpy(dtype=float)
    pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)

    metrics = evaluate_regression_full(y_true_log, pred_log, actual_usd, pred_usd)
    metrics["Training_Time_Seconds"] = training_time
    metrics["Generation_Time_Seconds"] = gen_time
    metrics["Parse_Success_Rate"] = float(valid_mask.mean())

    print(f"[{run_label}] RMSE_log={metrics['RMSE_log']:.4f} MAE_log={metrics['MAE_log']:.4f} "
          f"R2_log={metrics['R2_log']:.4f} parse_rate={metrics['Parse_Success_Rate']*100:.1f}%")

    return metrics, trainer.model, pred_log, valid_mask, generated_texts


---
# Optuna study (resumable via SQLite -- re-running this cell after a session death
# will skip already-completed trials and continue toward `N_TRIALS`)

Each trial trains on the **tuning subsample** (`tuning_train_df`), evaluates on `val_df`
(never on test), with a **fixed 2 epochs** and only LoRA/optimizer hyperparameters searched.


In [ ]:
def objective(trial):
    params = {
        "r": trial.suggest_categorical("r", [8, 16, 32]),
        "lora_alpha": trial.suggest_categorical("lora_alpha", [16, 32, 64]),
        "lora_dropout": trial.suggest_categorical("lora_dropout", [0.0, 0.05, 0.1]),
        "learning_rate": trial.suggest_categorical("learning_rate", [5e-5, 1e-4, 2e-4]),
        "num_train_epochs": FIXED_EPOCHS_PER_TRIAL,   # fixed, not searched -- keeps trials fast
        "train_batch_size": trial.suggest_categorical("train_batch_size", [4, 8]),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [2, 4]),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 0.01]),
        "warmup_ratio": trial.suggest_categorical("warmup_ratio", [0.0, 0.03]),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
    }

    trial_output_dir = f"{OPTUNA_WORKDIR}/trial_{trial.number}"
    trial_epoch_csv = f"{trial_output_dir}_epoch_metrics.csv"
    os.makedirs(os.path.dirname(trial_epoch_csv) or ".", exist_ok=True)

    metrics, model, _, _, _ = train_and_eval(
        params,
        train_df_local=tuning_train_df,
        eval_df_local=val_df,
        output_dir=trial_output_dir,
        run_label=f"trial_{trial.number}",
        epoch_metrics_csv=trial_epoch_csv,
    )

    del model
    cleanup_gpu()

    for key in ["MAE_log", "MSE_log", "R2_log", "MAE_USD", "MSE_USD", "RMSE_USD", "R2_USD", "RMSLE", "Parse_Success_Rate"]:
        trial.set_user_attr(key, metrics[key])

    return metrics["RMSE_log"]


In [ ]:
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"tinyllama_1_1b_{SERIALIZATION}_lora_optuna",
    storage=f"sqlite:///{STUDY_DB_PATH}",   # persists across session restarts
    load_if_exists=True,                    # re-running this cell resumes, doesn't restart
)

remaining_trials = N_TRIALS - len(study.trials)
if remaining_trials > 0:
    print(f"Running {remaining_trials} more trial(s) (already completed: {len(study.trials)})")
    study.optimize(objective, n_trials=remaining_trials)
else:
    print(f"All {N_TRIALS} trials already completed in a previous session -- skipping optimize().")

print("Best trial:", study.best_trial.number)
print("Best validation RMSE_log:", study.best_value)
print("Best params:", study.best_params)

trials_df = study.trials_dataframe()
trials_df.to_csv(f"{OUTPUT_DIR}/optuna_trials.csv", index=False)
display(trials_df.sort_values("value").head(10))


---
# Final retrain: best config on full train+val, evaluate on test **once**


In [ ]:
best_params = study.best_params.copy()
best_params["num_train_epochs"] = FIXED_EPOCHS_PER_TRIAL
trainval_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

tuned_epoch_csv = f"{OUTPUT_DIR}/tuned_epoch_metrics.csv"

tuned_metrics, tuned_model, tuned_pred_log, tuned_valid_mask, tuned_gen_texts = train_and_eval(
    best_params,
    train_df_local=trainval_df,
    eval_df_local=test_df,
    output_dir=f"{OUTPUT_DIR}/tuned_checkpoints",
    run_label="TUNED/test",
    epoch_metrics_csv=tuned_epoch_csv,
)

# Model saved in HF/safetensors format -- ready for later XAI use
tuned_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved tuned model to:", FINAL_MODEL_DIR)

tuned_result_row = {
    "Model": f"TinyLlama-1.1B-LoRA-Optuna-{SERIALIZATION}",
    "Serialization": SERIALIZATION,
    "Stage": "optuna_tuned",
    **best_params,
    **tuned_metrics,
}
tuned_final_df = pd.DataFrame([tuned_result_row])
tuned_final_df.to_csv(f"{OUTPUT_DIR}/tuned_final_metrics.csv", index=False)
display(tuned_final_df)


## Combine baseline vs tuned comparison (baseline metrics loaded from the input notebook)

In [ ]:
baseline_final_df = pd.read_csv(f"{BASELINE_INPUT_DIR}/baseline_final_metrics.csv")

comparison_df = pd.concat([baseline_final_df, tuned_final_df], ignore_index=True, sort=False)
display(comparison_df.round(6))
comparison_df.to_csv(f"{OUTPUT_DIR}/baseline_vs_tuned_{SERIALIZATION}.csv", index=False)

tuned_pred_df = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "actual_log_target": test_df["target"].to_numpy(),
    "predicted_log_target": tuned_pred_log,
    "raw_generated_text": tuned_gen_texts,
    "numeric_parse_valid": tuned_valid_mask,
    "actual_usd": test_df["target_usd"].to_numpy(),
    "predicted_usd": np.clip(np.expm1(tuned_pred_log), 0, None),
})
tuned_pred_df.to_csv(f"{OUTPUT_DIR}/tuned_predictions.csv", index=False)

print("Saved:")
print(f"  baseline_vs_tuned_{SERIALIZATION}.csv")
print("  tuned_predictions.csv")
print("  optuna_trials.csv")
print("  tuned_epoch_metrics.csv")


---
# Notes

- **Resuming after a session death**: re-add this same notebook's own prior output as an
  input (or just rerun in-place if Kaggle preserved `/kaggle/working/`), then rerun from the
  Optuna study cell — `load_if_exists=True` skips finished trials, and `resume_from_checkpoint`
  picks up any trial that died mid-epoch.
- **Runtime budget**: 5 trials × 2 epochs × ~5,500-row subsample should be substantially
  faster than a larger exploratory run. If it's still tight, drop `N_TRIALS` to 3-4 or
  `TUNING_SUBSAMPLE_SIZE` further.
- **Comparability**: trials and the tuned final model are evaluated on the same `val_df`/
  `test_df` IDs as the baseline notebook (via `shared_config.json`), so `RMSE_log` etc. are
  directly comparable across both notebooks' outputs.
- **Dependency on baseline notebook**: this notebook expects `shared_config.json` to contain
  `train_file` and `test_file` keys (not just `train_ids`/`val_ids`) -- confirm your
  `tinyllama_baseline.ipynb` writes those two keys, matching this notebook's Qwen counterpart.
